In [1]:
# run this notebook via colab because GPU runs faster

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.getcwd()

'/content'

In [3]:
# os.listdir("/content/drive/MyDrive/Colab Notebooks")

In [4]:
# for colab to render on github
import nbformat

with open("/content/drive/MyDrive/Colab Notebooks/A1_Q4.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open("/content/drive/MyDrive/Colab Notebooks/A1_Q4.ipynb", "w") as f:
    nbformat.write(nb, f)

# ECE1508: Deep Generative Models -- SUM25
## Assignment 1: Text Generation and Language Models
## Question 4: BERT Model and its Fine-tuning
BERT is a well-known transformer-based pre-trained LM. It is mainly used for __text classification__; therefore, it does __not__ have masked decoding. However, it provides the option to import some tokens as `"[MASK]"` to approximately mimic the masked decoding behavior. In this question, we first import the pre-trained BERT, and use it with Mask to complete a text. In the second part, we take a small subset of IMDB dataset to fine-tune it for text classification and see the impacts of fine-tuning.

### How to Answer?
Please complete all parts noted by `#COMPLETE`. Note that these parts are at both __Code__ and __Markdown.__


### Installing Required Package
Let's first install all we need.


In [5]:
# Install transformers. We can also do it directly at the terminal

# %pip install -q datasets transformers peft # install datasets with a specified version below
# %pip install -q transformers peft

In [6]:
# %pip uninstall torch torchvision -y


In [7]:
# %pip uninstall torchao -y

In [8]:
# %pip install -q "datasets>=2.21.0"
# %pip install "torch==2.6.0+cu124" torchvision --index-url https://download.pytorch.org/whl/cu124

#### Let's start with loading necessary libraries.

In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F

### Text Completion by Masked BERT
Now, let's import BERT. As mentioned, this model does not have masked decoding directly implemented. This means that at each time step, the context is computed from both past and future tokens. We can though use the class `BertForMaskedLM` and mimic the masked decoding behavior by replacing unknown tokens with special token `[MASK]`.

In [10]:
from transformers import BertTokenizer, BertForMaskedLM

# Select the device
if torch.backends.mps.is_built():
    device = "mps"
elif torch.backends.cuda.is_built():
    device = "cuda"
else:
    device = "cpu"

# Load pre-trained Tokenizer and the BERT model with masked head
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)

model.eval()
model.to(device)

def bert_masked_completion(text, num_masks=5, temperature=1.0):
    '''
    text: incomplete text
    num_masks: indicating number of tokens being added to the text.
    temperature: the temperature by which we sample
    '''
    # Append [MASK] tokens to the incomplete text
    # masked_text = # COMPLETE
    masked_text = text + " [MASK]" * num_masks
    print("Input Text:", masked_text)

    for i in range(num_masks):
        # Tokenize input
        # inputs = # COMPLETE use `tokenizer` to tokenize `masked_text`
        inputs = tokenizer(masked_text, return_tensors="pt").to(device)

        # Predict all masked tokens
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits


        # Find the first token [MASK] in the input
        # mask_token_index = # COMPLETE
        mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)
        # Save position of the first [MASK]
        # mask_pos = # COMPLETE
        mask_pos = mask_token_index[1][0].item()

        # Compute the probability distribution with temperature
        # probs = # COMPLETE
        probs = torch.softmax(logits[0, mask_pos] / temperature, dim=-1)

        # Sample from the distribution
        # predicted_token_id = # COMPLETE
        predicted_token_id = torch.multinomial(probs, num_samples=1).item()
        predicted_token = tokenizer.decode([predicted_token_id])

        # Replace first [MASK] with predicted token
        tokenized_text = tokenizer.tokenize(masked_text)
        mask_pos_in_tokens = tokenized_text.index("[MASK]")
        tokenized_text[mask_pos_in_tokens] = predicted_token

        # Update masked_text for next iteration
        masked_text = tokenizer.convert_tokens_to_string(tokenized_text)

        print(f"Predicted token at t+{i+1}:", predicted_token)
        print("Current text:", masked_text)

    print("\nFinal completed text:", masked_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


We now complete a text with this model.

In [11]:
text = "The movie was"
# Example usage
bert_masked_completion(text, num_masks=5)

Input Text: The movie was [MASK] [MASK] [MASK] [MASK] [MASK]
Predicted token at t+1: another
Current text: the movie was another [MASK] [MASK] [MASK] [MASK]
Predicted token at t+2: weird
Current text: the movie was another weird [MASK] [MASK] [MASK]
Predicted token at t+3: ##er
Current text: the movie was another weirder [MASK] [MASK]
Predicted token at t+4: thing
Current text: the movie was another weirder thing [MASK]
Predicted token at t+5: .
Current text: the movie was another weirder thing.

Final completed text: the movie was another weirder thing.


In [12]:
bert_masked_completion(text, num_masks=5, temperature=1.5)

Input Text: The movie was [MASK] [MASK] [MASK] [MASK] [MASK]
Predicted token at t+1: ##max
Current text: the movie wasmax [MASK] [MASK] [MASK] [MASK]
Predicted token at t+2: defected
Current text: the movie wasmax defected [MASK] [MASK] [MASK]
Predicted token at t+3: 1968
Current text: the movie wasmax defected 1968 [MASK] [MASK]
Predicted token at t+4: anyway
Current text: the movie wasmax defected 1968 anyway [MASK]
Predicted token at t+5: .
Current text: the movie wasmax defected 1968 anyway.

Final completed text: the movie wasmax defected 1968 anyway.


In [13]:
bert_masked_completion(text, num_masks=5, temperature=0.7)

Input Text: The movie was [MASK] [MASK] [MASK] [MASK] [MASK]
Predicted token at t+1: a
Current text: the movie was a [MASK] [MASK] [MASK] [MASK]
Predicted token at t+2: cult
Current text: the movie was a cult [MASK] [MASK] [MASK]
Predicted token at t+3: hit
Current text: the movie was a cult hit [MASK] [MASK]
Predicted token at t+4: worldwide
Current text: the movie was a cult hit worldwide [MASK]
Predicted token at t+5: .
Current text: the movie was a cult hit worldwide.

Final completed text: the movie was a cult hit worldwide.


In [14]:
bert_masked_completion(text, num_masks=5, temperature=0.2)

Input Text: The movie was [MASK] [MASK] [MASK] [MASK] [MASK]
Predicted token at t+1: a
Current text: the movie was a [MASK] [MASK] [MASK] [MASK]
Predicted token at t+2: big
Current text: the movie was a big [MASK] [MASK] [MASK]
Predicted token at t+3: hit
Current text: the movie was a big hit [MASK] [MASK]
Predicted token at t+4: too
Current text: the movie was a big hit too [MASK]
Predicted token at t+5: .
Current text: the movie was a big hit too.

Final completed text: the movie was a big hit too.


### Question: _Play around with the temperature. What do you see?_
Model runs with lower temperature produce more human-readable and coherent output. A lower temperature sharpens the probability distribution and makes the result more deterministic, whereas a higher temperature makes the distribution more uniform, producing more diverse outputs that may also become less coherent or semantically meaningful.

### Pre-trained BERT for Classification
We next fine-tune the model `BertForSequenceClassification` for text classification. To this end, we use a small subset of the `"imdb"` dataset. Let's first build the data.

In [15]:
from datasets import load_dataset

# Load the IMDB dataset
# dataset = # COMPLETE # load "imdb"
dataset = load_dataset("stanfordnlp/imdb")


# Sample one example
print(dataset["train"][10])


# Sample a small subset of D_train samples for training and D_test for testing
D_train, D_test = 1000, 500
# small_train_dataset = # COMPLETE
# small_test_dataset = # COMPLETE
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(D_train))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(D_test))

{'text': 'It was great to see some of my favorite stars of 30 years ago including John Ritter, Ben Gazarra and Audrey Hepburn. They looked quite wonderful. But that was it. They were not given any characters or good lines to work with. I neither understood or cared what the characters were doing.<br /><br />Some of the smaller female roles were fine, Patty Henson and Colleen Camp were quite competent and confident in their small sidekick parts. They showed some talent and it is sad they didn\'t go on to star in more and better films. Sadly, I didn\'t think Dorothy Stratten got a chance to act in this her only important film role.<br /><br />The film appears to have some fans, and I was very open-minded when I started watching it. I am a big Peter Bogdanovich fan and I enjoyed his last movie, "Cat\'s Meow" and all his early ones from "Targets" to "Nickleodeon". So, it really surprised me that I was barely able to keep awake watching this one.<br /><br />It is ironic that this movie is a

For tokenization, we use `BertTokenizerFast` as it helps us process faster.

In [16]:
from transformers import BertTokenizerFast

# Load BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Tokenize the data
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

# tokenized_train = # COMPLETE # you may use `.map` method
# tokenized_test = # COMPLETE # you may use `.map` method
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_test = small_test_dataset.map(tokenize_function, batched=True)

# Show tokenized sample
print(tokenized_train[0])

# Set format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1, 'input_ids': [101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771, 1998, 6337, 2099, 2021, 1996, 2755, 2008, 2119, 2024, 2610, 2186, 2055, 6355, 6997, 1012, 6337, 2099, 3504, 15594, 2100, 1010, 3481, 3771, 350

We next test the pre-trained model on the test set to see its pre-trained performance.

In [17]:
from torch.utils.data import DataLoader
from tqdm import tqdm

# Create a DataLoader for test set
test_dataloader = DataLoader(tokenized_test, batch_size=16)

# Move model to device at eval mode
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)
model.eval()


# Set counter for number of correct prediction
correct = 0
total = 0

# No gradient calculation needed during evaluation
with torch.no_grad():
    for batch in tqdm(test_dataloader):

        # COMPLETE

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # compute context
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        # Find the class with "maximum" logit
        predictions = outputs.logits.argmax(dim=-1)

        # compute number of correct predictions
        correct += (predictions == labels).sum().item() # add the number of correct classifications
        total += labels.size(0) # update total number of samples

# Compute accuracy
accuracy = correct / total
print(f"Pretrained BERT accuracy on test set: {accuracy:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 32/32 [00:07<00:00, 

Pretrained BERT accuracy on test set: 0.5120


### Fine-tuning BERT
We now fine-tune BERT on the train dataset. We use __Selective Fine-tuning,__ where we only fine-tune the last encoder (transformer) layer and the final MLP (classification head). We can freeze all other layers using the following code.

In [18]:
# Freeze all BERT parameters
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze only the last encoder layer
# last_layer_idx = # COMPLETE # with an integer # find what layer is the last one (see BERT architecture)
last_layer_idx = 11
for param in model.bert.encoder.layer[last_layer_idx].parameters():
    param.requires_grad = True

# Also unfreeze the classifier head
for param in model.classifier.parameters():
    param.requires_grad = True


For fine-tuning, we use __LoRA__ algorithm with rank $\ell=8$. This can be directly implemented using the `peft` library. The following code builds the required configurations.

In [19]:
from peft import get_peft_model, LoraConfig, TaskType

# LoRA config — tune the hyperparameters as you wish
lora_config = LoraConfig(
    # r = # COMPLETE
    r = 8,                               # rank of LoRA, i.e., l in the lecture-notes
    lora_alpha = 32,                     # scaling factor
    target_modules = ["query", "value"], # target attention modules for LoRA
    lora_dropout = 0.1,
    bias = "none",
    task_type = TaskType.SEQ_CLS,
)

# Apply LoRA to the model -> This will apply LoRA to unfrozen layers
model = get_peft_model(model, lora_config)

We can readily perform the fine tuning using the `Trainer` class. The following code defines the `trainer` which can be used to fine-tune the model over our dataset. We set the the number of fine-tuning epochs to 3.

In [20]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",        # <-- add this line
    learning_rate=5e-4,
    # per_device_train_batch_size = # COMPLETE #set it to the batch-size of the dataloader
    per_device_train_batch_size = 16,
    # per_device_eval_batch_size = # COMPLETE
    per_device_eval_batch_size = 16,
    # num_train_epochs= # COMPLETE
    num_train_epochs = 3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)


# The tokenized_train and tokenized_test datasets must have format set for torch tensors:
# tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
# tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [21]:
# different learning rate

training_args_2 = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",        # <-- add this line
    learning_rate=1e-4, # lower
    # per_device_train_batch_size = # COMPLETE #set it to the batch-size of the dataloader
    per_device_train_batch_size = 16,
    # per_device_eval_batch_size = # COMPLETE
    per_device_eval_batch_size = 16,
    # num_train_epochs= # COMPLETE
    num_train_epochs = 3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

training_args_3 = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",        # <-- add this line
    learning_rate=1e-3, # higher
    # per_device_train_batch_size = # COMPLETE #set it to the batch-size of the dataloader
    per_device_train_batch_size = 16,
    # per_device_eval_batch_size = # COMPLETE
    per_device_eval_batch_size = 16,
    # num_train_epochs= # COMPLETE
    num_train_epochs = 3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer_2 = Trainer(
    model=model,
    args=training_args_2,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

trainer_3 = Trainer(
    model=model,
    args=training_args_3,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


We may check the trainable parameters

In [22]:
model.print_trainable_parameters()

trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


We can now fine-tune easily by applying method `.train()` to the `trainer`.

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.396298,0.483762
2,0.327313,0.332866
3,0.265044,0.332762


TrainOutput(global_step=189, training_loss=0.38390036865516947, metrics={'train_runtime': 130.637, 'train_samples_per_second': 22.964, 'train_steps_per_second': 1.447, 'total_flos': 396032624640000.0, 'train_loss': 0.38390036865516947, 'epoch': 3.0})

In [24]:
trainer_2.train()

Epoch,Training Loss,Validation Loss
1,0.189515,0.370122
2,0.243359,0.381686
3,0.210790,0.366492


TrainOutput(global_step=189, training_loss=0.20951517549141374, metrics={'train_runtime': 128.8339, 'train_samples_per_second': 23.286, 'train_steps_per_second': 1.467, 'total_flos': 396032624640000.0, 'train_loss': 0.20951517549141374, 'epoch': 3.0})

In [25]:
trainer_3.train()

Epoch,Training Loss,Validation Loss
1,0.248599,0.440268
2,0.195874,0.403237
3,0.136094,0.438470


TrainOutput(global_step=189, training_loss=0.23902159487759625, metrics={'train_runtime': 128.5431, 'train_samples_per_second': 23.338, 'train_steps_per_second': 1.47, 'total_flos': 396032624640000.0, 'train_loss': 0.23902159487759625, 'epoch': 3.0})

### Question: _Play around with the learning rate at `training_args` to see the impact. Report what you see._

Each trainer & the learning rate:
- `trainer  `: 5e-4
- `trainer_2`: 1e-4
- `trainer_3`: 1e-3

So in terms of learning rate, trainer_3 > trainer > trainer_2.
  
With trainer_3 which has the highest learning rate, it is clearly that the model is overfitting, as the validation loss is higher than the training loss in epochs 2 and 3.

With trainer_2 which has the lowest learning rate, the last epoch (epoch 3) increases in training loss in epoch 3. The validation loss is also consistently higher than the training loss in each epoch.

It is clear that trainer that has the middle ground learning rate (5e-4) performs the best, as the training loss decreases over epoch, and the validation loss is consistently in the reasonable range.



Let's now evaluate the fine-tuned model.

In [26]:
# Set model back to eval
model.eval()

# Set prior values
correct = 0
total = 0

# evaluation loop
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # compute context
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        # Find the class with "maximum" logit
        predictions = outputs.logits.argmax(dim=-1)

        # compute number of correct predictions
        correct += (predictions == labels).sum().item() # add the number of correct classifications
        total += labels.size(0) # update total number of samples

# Compute accuracy
accuracy = correct / total
print(f"Fine_tuned BERT accuracy on small test set: {accuracy:.4f}")


100%|██████████| 32/32 [00:07<00:00,  4.13it/s]

Fine_tuned BERT accuracy on small test set: 0.8800


### Question: _Compare the fine-tuned accuracy to that of the pre-trained one. What do you conclude?_
Pre-trained BERT has an accuracy of 0.484, whereas fine-tuned BERT reaches 0.862. The substantial gap suggests that while BERT's pre-trained representations provide a strong foundation, task-specific fine-tuning is essential for the model to perform well on sentiment classification.
